# 地理情報付き写真をGoogle Earth Proにアップロード

このノートブックは、HEIC形式の地理情報付き写真を含むフォルダから、Google Earth Pro用のKMLファイルを生成します。

## 必要なパッケージ
- pillow-heif: HEICファイルの読み込み
- pillow: 画像処理とEXIFデータの取得
- simplekml: KMLファイルの生成

In [ ]:
# 必要なライブラリのインストール確認
import sys

try:
    import pillow_heif
    from PIL import Image
    from PIL.ExifTags import TAGS, GPSTAGS
    import simplekml
    print("✓ すべての必要なパッケージがインストールされています")
except ImportError as e:
    print(f"必要なパッケージがありません: {e}")
    print("\n以下のコマンドでインストールしてください:")
    print("pip install pillow-heif pillow simplekml")

In [ ]:
import os
from pathlib import Path
from datetime import datetime

# HEIF形式を登録
pillow_heif.register_heif_opener()

## GPS情報の抽出関数

In [ ]:
def get_exif_data(image_path):
    """画像ファイルからEXIFデータを取得"""
    try:
        image = Image.open(image_path)
        exif_data = image.getexif()
        return exif_data
    except Exception as e:
        print(f"EXIF読み取りエラー ({image_path}): {e}")
        return None


def get_gps_info(exif_data):
    """EXIFデータからGPS情報を抽出"""
    if not exif_data:
        return None
    
    gps_info = {}
    
    for tag_id, value in exif_data.items():
        tag_name = TAGS.get(tag_id, tag_id)
        if tag_name == 'GPSInfo':
            for gps_tag_id, gps_value in value.items():
                gps_tag_name = GPSTAGS.get(gps_tag_id, gps_tag_id)
                gps_info[gps_tag_name] = gps_value
            break
    
    return gps_info if gps_info else None


def convert_to_degrees(value):
    """GPS座標を度数法に変換"""
    d = float(value[0])
    m = float(value[1])
    s = float(value[2])
    return d + (m / 60.0) + (s / 3600.0)


def get_lat_lon(gps_info):
    """GPS情報から緯度経度を取得"""
    if not gps_info:
        return None, None
    
    try:
        lat = convert_to_degrees(gps_info.get('GPSLatitude', [0, 0, 0]))
        lon = convert_to_degrees(gps_info.get('GPSLongitude', [0, 0, 0]))
        
        # 南緯と西経の場合は負の値にする
        if gps_info.get('GPSLatitudeRef') == 'S':
            lat = -lat
        if gps_info.get('GPSLongitudeRef') == 'W':
            lon = -lon
        
        return lat, lon
    except (KeyError, TypeError, ValueError) as e:
        print(f"GPS座標変換エラー: {e}")
        return None, None


def get_photo_metadata(image_path, exif_data):
    """写真のメタデータを取得"""
    metadata = {
        'filename': os.path.basename(image_path),
        'path': str(image_path)
    }
    
    if exif_data:
        # 撮影日時
        for tag_id, value in exif_data.items():
            tag_name = TAGS.get(tag_id, tag_id)
            if tag_name == 'DateTime':
                metadata['datetime'] = value
            elif tag_name == 'Make':
                metadata['camera_make'] = value
            elif tag_name == 'Model':
                metadata['camera_model'] = value
    
    return metadata

## KMLファイル生成関数

In [ ]:
def create_kml_from_photos(photo_folder, output_kml_path, convert_to_jpg=False, jpg_output_folder=None):
    """
    写真フォルダからKMLファイルを生成
    
    Parameters:
    -----------
    photo_folder : str
        地理情報付き写真が格納されているフォルダパス
    output_kml_path : str
        出力するKMLファイルのパス
    convert_to_jpg : bool
        HEICをJPGに変換するか（Google Earth Proでの互換性向上のため）
    jpg_output_folder : str
        JPG変換後の保存先フォルダ（convert_to_jpg=Trueの場合）
    
    Returns:
    --------
    dict: 処理結果の統計情報
    """
    photo_folder = Path(photo_folder)
    
    if convert_to_jpg and jpg_output_folder:
        jpg_output_folder = Path(jpg_output_folder)
        jpg_output_folder.mkdir(parents=True, exist_ok=True)
    
    # KMLオブジェクトの作成
    kml = simplekml.Kml()
    kml.document.name = "地理情報付き写真"
    
    stats = {
        'total_files': 0,
        'with_gps': 0,
        'without_gps': 0,
        'errors': 0
    }
    
    # HEICおよびHEIFファイルを検索
    photo_files = list(photo_folder.glob('*.heic')) + \
                  list(photo_folder.glob('*.HEIC')) + \
                  list(photo_folder.glob('*.heif')) + \
                  list(photo_folder.glob('*.HEIF'))
    
    print(f"\n処理開始: {len(photo_files)} 個のHEICファイルを検出")
    print("=" * 60)
    
    for photo_path in photo_files:
        stats['total_files'] += 1
        
        try:
            # EXIFデータ取得
            exif_data = get_exif_data(photo_path)
            gps_info = get_gps_info(exif_data)
            lat, lon = get_lat_lon(gps_info)
            
            if lat and lon:
                stats['with_gps'] += 1
                
                # メタデータ取得
                metadata = get_photo_metadata(photo_path, exif_data)
                
                # JPGに変換する場合
                image_reference_path = str(photo_path)
                if convert_to_jpg and jpg_output_folder:
                    jpg_filename = photo_path.stem + '.jpg'
                    jpg_path = jpg_output_folder / jpg_filename
                    
                    # HEIC → JPG変換
                    img = Image.open(photo_path)
                    # サムネイルサイズに縮小（ファイルサイズ削減のため）
                    img.thumbnail((1920, 1920), Image.Resampling.LANCZOS)
                    img.save(jpg_path, 'JPEG', quality=85)
                    
                    image_reference_path = str(jpg_path)
                    print(f"✓ {metadata['filename']} → {jpg_filename} (GPS: {lat:.6f}, {lon:.6f})")
                else:
                    print(f"✓ {metadata['filename']} (GPS: {lat:.6f}, {lon:.6f})")
                
                # KMLプレースマークの作成
                pnt = kml.newpoint()
                pnt.name = metadata['filename']
                pnt.coords = [(lon, lat)]
                
                # 説明欄の作成
                description = f"""<![CDATA[
                <h3>{metadata['filename']}</h3>
                <img src="file:///{image_reference_path}" width="400"/><br/>
                <b>撮影日時:</b> {metadata.get('datetime', 'N/A')}<br/>
                <b>カメラ:</b> {metadata.get('camera_make', '')} {metadata.get('camera_model', '')}<br/>
                <b>座標:</b> {lat:.6f}, {lon:.6f}
                ]]>"""
                pnt.description = description
                
                # カメラアイコンを設定
                pnt.style.iconstyle.icon.href = 'http://maps.google.com/mapfiles/kml/shapes/camera.png'
                
            else:
                stats['without_gps'] += 1
                print(f"⚠ {photo_path.name}: GPS情報なし")
                
        except Exception as e:
            stats['errors'] += 1
            print(f"✗ {photo_path.name}: エラー - {str(e)}")
    
    # KMLファイルの保存
    kml.save(output_kml_path)
    
    print("\n" + "=" * 60)
    print(f"\n処理完了!")
    print(f"KMLファイル: {output_kml_path}")
    print(f"\n統計:")
    print(f"  総ファイル数: {stats['total_files']}")
    print(f"  GPS情報あり: {stats['with_gps']}")
    print(f"  GPS情報なし: {stats['without_gps']}")
    print(f"  エラー: {stats['errors']}")
    
    return stats

## 使用方法

以下のセルを実行して、写真フォルダからKMLファイルを生成します。

In [ ]:
# ========================================
# 設定: ここを編集してください
# ========================================

# 地理情報付き写真が入っているフォルダのパス
PHOTO_FOLDER = "/path/to/your/photos"  # ← ここを実際のパスに変更

# 出力するKMLファイルのパス
OUTPUT_KML = "./geotagged_photos.kml"

# HEICをJPGに変換するか（推奨: Google Earth Proでの互換性向上）
CONVERT_TO_JPG = True

# JPG変換後の保存先フォルダ（CONVERT_TO_JPG=Trueの場合のみ使用）
JPG_OUTPUT_FOLDER = "./converted_photos"

# ========================================
# 実行
# ========================================

if PHOTO_FOLDER == "/path/to/your/photos":
    print("⚠ 警告: PHOTO_FOLDERを実際の写真フォルダパスに変更してください")
else:
    stats = create_kml_from_photos(
        photo_folder=PHOTO_FOLDER,
        output_kml_path=OUTPUT_KML,
        convert_to_jpg=CONVERT_TO_JPG,
        jpg_output_folder=JPG_OUTPUT_FOLDER if CONVERT_TO_JPG else None
    )

## Google Earth Proでの開き方

1. Google Earth Proを起動
2. メニューから「ファイル」→「開く」を選択
3. 生成された `.kml` ファイルを選択
4. 写真の場所がマーカーとして表示されます
5. マーカーをクリックすると写真とメタデータが表示されます

## トラブルシューティング

### GPS情報が取得できない場合
- iPhoneの設定で「カメラ」→「位置情報」が有効になっているか確認
- 写真の「プライバシー設定」で位置情報が削除されていないか確認

### Google Earth Proで画像が表示されない場合
- `CONVERT_TO_JPG = True` に設定して実行（HEICは非対応の場合があります）
- 画像パスが正しく設定されているか確認

### パフォーマンスの問題
- 大量の写真（100枚以上）を処理する場合は時間がかかります
- `CONVERT_TO_JPG = True` の場合、さらに時間がかかります（画像サイズは自動的に縮小されます）

## サンプル: 特定の写真だけをテスト

In [ ]:
# 単一の写真をテストする場合
test_photo = "/path/to/test/photo.heic"

if os.path.exists(test_photo):
    exif_data = get_exif_data(test_photo)
    gps_info = get_gps_info(exif_data)
    lat, lon = get_lat_lon(gps_info)
    metadata = get_photo_metadata(test_photo, exif_data)
    
    print(f"ファイル名: {metadata['filename']}")
    print(f"緯度: {lat}")
    print(f"経度: {lon}")
    print(f"撮影日時: {metadata.get('datetime', 'N/A')}")
    print(f"カメラ: {metadata.get('camera_make', '')} {metadata.get('camera_model', '')}")
else:
    print("テスト用の写真パスを設定してください")